In [ ]:
import glob
import os
import subprocess
import time

sen_files = sorted(glob.glob("langpro-datasets/*_sen.pl"))
total = len(sen_files)
print(f"Found {total} datasets to prove.")

gstart = time.time()

c = 0
with open("proving.log", "w") as log:
    for sen_path in sen_files:
        start = time.time()
        basename = os.path.basename(sen_path).replace("_sen.pl", "")
        ccg_path = sen_path.replace("_sen.pl", "_llm_ccg.pl")
        result_path = os.path.join("results", f"{basename}_pred.txt")

        if not os.path.exists(ccg_path):
            print(f"SKIP {basename}: no CCG file found")
            continue

        cmd = [
            "swipl",
            "-g",
            f"parList([prprb, ral(50), aall, wn_ant, wn_sim, wn_der, constchk, waif('{result_path}')]), entail_all, halt",
            "-f",
            "../LangPro/prolog/main.pl",
            "../LangPro/WNProlog/wn.pl",
            sen_path,
            ccg_path,

        ]

        print(f"Proving {basename}...")
        subprocess.run(cmd, stderr=log, stdout=subprocess.DEVNULL)
        c += 1
        print(f"{c}/{total} written to {result_path}, took {time.time() - start:.1f}s")
total_elapsed = time.time() - gstart
print(f"\nProved {c} datasets in {total_elapsed:.1f}s (avg {total_elapsed/c:.1f}s/dataset)")